In [2]:
import pandas as pd
import joblib
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, f1_score

# Load splits & encoder
train_df = pd.read_csv('../data/processed/train.csv')
val_df = pd.read_csv('../data/processed/val.csv')
test_df = pd.read_csv('../data/processed/test.csv')
label_encoder = joblib.load('../model/artifacts/label_encoder.joblib')

# Vectorize features
tfidf = TfidfVectorizer(max_features=2500, ngram_range=(1, 2))
X_train = tfidf.fit_transform(train_df['cleaned_description'])
X_val = tfidf.transform(val_df['cleaned_description'])
X_test = tfidf.transform(test_df['cleaned_description'])

y_train = train_df['label']
y_val = val_df['label']
y_test = test_df['label']

# Train XGBoost
model = XGBClassifier(
    n_estimators=150,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    eval_metric='mlogloss'
)

model.fit(X_train, y_train)

# Evaluate model
test_preds = model.predict(X_test)
macro_f1 = f1_score(y_test, test_preds, average='macro')

print(f"Test Set Macro F1-Score: {macro_f1:.4f}\n")
print(classification_report(y_test, test_preds, target_names=label_encoder.classes_))

# Save artifacts
joblib.dump(tfidf, '../model/artifacts/tfidf_vectorizer.joblib')
joblib.dump(model, '../model/artifacts/xgboost_model.joblib')

metrics = {"macro_f1": float(macro_f1), "classes": label_encoder.classes_.tolist()}
with open('../model/artifacts/model_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=4)

print("Model artifacts saved to model/artifacts/")

Test Set Macro F1-Score: 0.9994

               precision    recall  f1-score   support

       exempt       1.00      1.00      1.00       135
outside_scope       1.00      1.00      1.00       135
 reduced_rate       1.00      1.00      1.00        90
standard_rate       1.00      1.00      1.00       386
   zero_rated       1.00      1.00      1.00       270

     accuracy                           1.00      1016
    macro avg       1.00      1.00      1.00      1016
 weighted avg       1.00      1.00      1.00      1016

Model artifacts saved to model/artifacts/
